In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
import nltk
from nltk.stem.snowball import SnowballStemmer
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [13]:
# Скачиваем необходимые ресурсы NLTK
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

# Инициализируем стеммер
stemmer = SnowballStemmer("russian")

# Определяем факторы привлекательности
FACTORS = {
    'экономические': {
        'keywords': [
            'зарплата', 'зарплаты', 'заработок', 'заработная плата', 'оклад', 'выплата', 'стипендия',
            'трудоустройство', 'трудоустроиться', 'вакансия', 'работа', 'рабочее место', 
            'карьера', 'карьерный рост', 'должность',
            'востребованность', 'востребованный', 'спрос', 'дефицит кадров', 'гарантия трудоустройства',
            'доход', 'доходы', 'деньги', 'зарплатный'
        ],
        'description': 'Экономическая привлекательность: зарплаты, трудоустройство, востребованность'
    },
    'социально_психологические': {
        'keywords': [
            'престиж', 'престижный', 'престижная', 'уважение', 'статус', 'известный',
            'независимость', 'самостоятельность', 'взрослая', 'жизнь', 
            'стереотип', 'миф', 'грязная работа', 'современные условия', 'комфортные',
            'родители', 'семья', 'поддержка',
            'друзья', 'сверстники', 'команда', 'коллектив',
            'гордость', 'почет', 'социальный лифт'
        ],
        'description': 'Социально-психологические факторы: престиж, независимость, поддержка'
    },
    'образовательные': {
        'keywords': ['практика', 'мастерская практика', 'аккредитация', 'лицензия',
            'опытный', 'высококвалифицированный', 'мастер', 'победа', 'мастерство', 'лаборатория', 'новый', 'нацпроект',
            'трудоустройство', 'гарантия', 'сотрудничество', 'соцпартнер', 'партнер', 'карьера'
        ],
        'description': 'Образовательные факторы: условия обучения, поступление, карьера'
    },
    'взаимодействие_с_бизнесом': {
        'keywords': [
            'базовое предприятие', 'завод', 'фабрика', 'работодатель', 'партнер', 'соцпартнер', 'бизнес',
            'целевое', 'договор', 'рабочее', 'комбинат', 'вуз', 'НПО',
            'дуальное', 'профстандарт', 'профессионалитет',
            'трудоустройство', 'работа', 'выпускник',
            'контракт', 'компания', 'спонсорство', 'экскурсия'
        ],
        'description': 'Взаимодействие с бизнесом: партнерства, экскурсии, работа'
    },
    'медийная_эффективность': {
        'description': 'Рассчитывается отдельно по колонке ER Post'
    }
}

# Получаем стемы для всех ключевых слов
factor_stems = {}
print("="*80)
print("СТЕМЫ КЛЮЧЕВЫХ СЛОВ:")
print("="*80)

for factor_name, factor_data in FACTORS.items():
    if factor_name != 'медийная_эффективность':
        stems = set()
        for keyword in factor_data['keywords']:
            words = keyword.lower().split()
            for word in words:
                if '-' in word:
                    parts = word.split('-')
                    for part in parts:
                        stems.add(stemmer.stem(part))
                else:
                    stems.add(stemmer.stem(word))
        factor_stems[factor_name] = stems
        print(f"Фактор '{factor_name}': {len(stems)} стемов")

# Информация о файлах
FILE_INFO = {
    'ЗТТиЭ': {
        '2023': 'ZTTiE-23.xlsx',
        '2024': 'ZTTiE-24.xlsx',
        '2025': 'ZTTiE-25.xlsx'
    },
    'Профессионалитет ЗТТиЭ': {
        '2023': 'Profi-23.xlsx',
        '2024': 'Profi-24.xlsx',
        '2025': 'Profi-25.xlsx'
    },
    'Аносовский колледж': {
        '2023': 'Zlatik-23.xlsx',
        '2024': 'Zlatik-24.xlsx',
        '2025': 'Zlatik-25.xlsx'
    }
}

def get_text_stems(text):
    """Извлекает стемы из текста"""
    if pd.isna(text) or not isinstance(text, str):
        return set()
    text_lower = text.lower()
    words = re.findall(r'[а-яё]+(?:-[а-яё]+)?', text_lower)
    stems = set()
    for word in words:
        if '-' in word:
            parts = word.split('-')
            for part in parts:
                stems.add(stemmer.stem(part))
        else:
            stems.add(stemmer.stem(word))
    return stems

def process_file(filepath, source_name, year):
    """
    Обрабатывает один файл: анализирует факторы в колонке Text
    """
    try:
        df = pd.read_excel(filepath)
    except FileNotFoundError:
        print(f"  Файл {filepath} не найден. Пропускаем.")
        return None
    
    print(f"\n  Файл {filepath} загружен. Постов: {len(df)}")
    
    # Проверяем наличие колонки Text
    if 'Text' not in df.columns:
        print(f"  ВНИМАНИЕ: В файле {filepath} отсутствует колонка Text")
        return None
    
    # Инициализируем счетчики факторов
    factor_counts = {factor: 0 for factor in FACTORS.keys() if factor != 'медийная_эффективность'}
    
    # Анализируем каждый пост
    total_valid = 0
    for idx, row in df.iterrows():
        text = str(row['Text']) if pd.notna(row['Text']) else ''
        if len(text) < 10:  # пропускаем слишком короткие тексты
            continue
        
        total_valid += 1
        text_stems = get_text_stems(text)
        
        # Проверяем каждый фактор
        for factor_name, stems_set in factor_stems.items():
            if text_stems & stems_set:  # если есть пересечение
                factor_counts[factor_name] += 1
    
    # Считаем средний ER Post
    avg_er = 0
    if 'ER Post' in df.columns:
        er_values = pd.to_numeric(df['ER Post'], errors='coerce')
        avg_er = er_values.mean()
        if pd.isna(avg_er):
            avg_er = 0
    
    result = {
        'source': source_name,
        'year': year,
        'total_posts': total_valid,
        'factor_counts': factor_counts,
        'avg_er_post': avg_er
    }
    
    # Выводим результаты
    print(f"\n  РЕЗУЛЬТАТЫ {source_name} ({year}):")
    print(f"    Всего постов с текстом: {total_valid}")
    print(f"    ER средний: {avg_er:.3f}")
    for factor, count in factor_counts.items():
        if count > 0:
            percent = (count / total_valid) * 100 if total_valid > 0 else 0
            print(f"    - {factor}: {count} постов ({percent:.1f}%)")
    
    return result

def aggregate_ztte_results(results):
    """Агрегирует результаты пабликов ЗТТиЭ и Профессионалитет ЗТТиЭ"""
    aggregated = {}
    
    for year in ['2023', '2024', '2025']:
        ztte_results = [r for r in results if r and r['source'] == 'ЗТТиЭ' and r['year'] == year]
        prof_results = [r for r in results if r and r['source'] == 'Профессионалитет ЗТТиЭ' and r['year'] == year]
        
        if ztte_results and prof_results:
            ztte = ztte_results[0]
            prof = prof_results[0]
            
            # Суммируем счетчики факторов
            total_factors = {}
            for factor in FACTORS.keys():
                if factor != 'медийная_эффективность':
                    total_factors[factor] = (
                        ztte['factor_counts'].get(factor, 0) + 
                        prof['factor_counts'].get(factor, 0)
                    )
            
            # Общее количество постов
            total_posts = ztte['total_posts'] + prof['total_posts']
            
            # Средний ER Post
            weighted_er = (
                (ztte['avg_er_post'] * ztte['total_posts'] + 
                 prof['avg_er_post'] * prof['total_posts']) / total_posts
            ) if total_posts > 0 else 0
            
            aggregated[year] = {
                'source': 'ЗТТиЭ (сумма)',
                'year': year,
                'total_posts': total_posts,
                'factor_counts': total_factors,
                'avg_er_post': weighted_er
            }
            print(f"\nАгрегировано ЗТТиЭ ({year}): {total_posts} постов")
    
    return aggregated

def calculate_factor_scores(data):
    """
    ПРАВИЛЬНЫЙ РАСЧЕТ:
    для каждого фактора: количество постов с этим фактором / общее количество постов * 100
    Это дает долю постов (в процентах), посвященных каждому фактору
    """
    scores = {}
    for key, item in data.items():
        score_item = {
            'source': item['source'],
            'year': item['year'],
            'factor_scores': {},
            'total_posts': item['total_posts']
        }
        
        # Рассчитываем процент для каждого фактора
        for factor, count in item['factor_counts'].items():
            if item['total_posts'] > 0:
                percent = (count / item['total_posts']) * 100
                score_item['factor_scores'][factor] = round(percent, 1)
            else:
                score_item['factor_scores'][factor] = 0
        
        # Добавляем медийную эффективность (как есть, без нормирования)
        score_item['factor_scores']['медийная_эффективность'] = round(item['avg_er_post'], 3)
        
        scores[key] = score_item
    
    return scores

def plot_comparison_by_year(df, year):
    """Строит сравнительную столбчатую диаграмму для указанного года"""
    ztte_year = df[(df['Учреждение'] == 'ЗТТиЭ') & (df['Год'] == year)]
    anosov_year = df[(df['Учреждение'] == 'Аносовский колледж') & (df['Год'] == year)]
    
    if ztte_year.empty or anosov_year.empty:
        print(f"  Нет данных для {year} года")
        return
    
    factors_pct = ['Экономические, %', 'Социально-психол., %', 'Образовательные, %', 'Связь с бизнесом, %']
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    x = np.arange(len(factors_pct))
    width = 0.35
    
    ztte_vals = [ztte_year.iloc[0][f] for f in factors_pct]
    anosov_vals = [anosov_year.iloc[0][f] for f in factors_pct]
    
    bars1 = ax.bar(x - width/2, ztte_vals, width, label='ЗТТиЭ', color='coral')
    bars2 = ax.bar(x + width/2, anosov_vals, width, label='Аносовский колледж', color='skyblue')
    
    ax.set_xlabel('Факторы')
    ax.set_ylabel('Доля постов, %')
    ax.set_title(f'Сравнение факторов привлекательности в {year} году')
    ax.set_xticks(x)
    ax.set_xticklabels([f.replace(', %', '') for f in factors_pct], rotation=45, ha='right')
    ax.legend()
    
    # Добавляем значения на столбцы
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 0:
                ax.annotate(f'{height:.1f}%',
                           xy=(bar.get_x() + bar.get_width()/2, height),
                           xytext=(0, 3), textcoords="offset points",
                           ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    filename = f'сравнение_факторов_{year}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  График сохранен: {filename}")

def plot_ztte_business_bubble(df_results, raw_results, factor_stems_data):
    """
    Строит пузырьковые диаграммы для фактора "Связь с бизнесом" по каждому ключевому слову
    """
    
    # Получаем сырые данные для ЗТТиЭ и Профессионалитет ЗТТиЭ
    ztte_raw = {}
    prof_raw = {}
    
    for r in raw_results:
        if r and r['source'] == 'ЗТТиЭ':
            ztte_raw[r['year']] = r
        elif r and r['source'] == 'Профессионалитет ЗТТиЭ':
            prof_raw[r['year']] = r
    
    if not ztte_raw or not prof_raw:
        print("  Нет сырых данных для пузырьковых диаграмм")
        return
    
    # Получаем оригинальные ключевые слова для фактора "Связь с бизнесом"
    business_keywords = FACTORS['взаимодействие_с_бизнесом']['keywords']
    
    # Предварительно загружаем и кешируем стемы для всех текстов по годам
    print("\n  Предварительная загрузка и стемминг текстов...")
    texts_cache = {}
    
    for year in ['2023', '2024', '2025']:
        if year not in ztte_raw or year not in prof_raw:
            continue
            
        ztte_file = FILE_INFO['ЗТТиЭ'][year]
        prof_file = FILE_INFO['Профессионалитет ЗТТиЭ'][year]
        
        try:
            df_ztte = pd.read_excel(ztte_file)
            df_prof = pd.read_excel(prof_file)
        except FileNotFoundError:
            print(f"  Файлы за {year} год не найдены")
            continue
        
        # Собираем все тексты для года
        all_texts = []
        if 'Text' in df_ztte.columns:
            all_texts.extend(df_ztte['Text'].fillna('').astype(str).tolist())
        if 'Text' in df_prof.columns:
            all_texts.extend(df_prof['Text'].fillna('').astype(str).tolist())
        
        # Кешируем стемы для каждого текста
        text_stems_list = []
        for text in all_texts:
            if len(text) >= 10:
                text_stems_list.append(get_text_stems(text))
            else:
                text_stems_list.append(set())
        
        texts_cache[year] = {
            'texts': all_texts,
            'stems_list': text_stems_list
        }
        print(f"    Загружено {len(all_texts)} текстов для {year} года")
    
    # Для каждого года строим график
    for year in ['2023', '2024', '2025']:
        if year not in texts_cache:
            print(f"  Нет данных за {year} год для пузырьковой диаграммы")
            continue
        
        print(f"\n  Обработка {year} года...")
        cache = texts_cache[year]
        
        # Подсчитываем частоту каждого ключевого слова
        keyword_stats = []
        
        for keyword in business_keywords:
            # Получаем стемы для ключевого слова
            keyword_stems = set()
            words = keyword.lower().split()
            for word in words:
                if '-' in word:
                    parts = word.split('-')
                    for part in parts:
                        keyword_stems.add(stemmer.stem(part))
                else:
                    keyword_stems.add(stemmer.stem(word))
            
            # Считаем посты, где встречается это ключевое слово
            posts_with_keyword = 0
            total_mentions = 0
            
            for i, text_stems in enumerate(cache['stems_list']):
                if text_stems & keyword_stems:
                    posts_with_keyword += 1
                    # Считаем упоминания (проходим по исходному тексту)
                    text = cache['texts'][i].lower()
                    mention_count = 0
                    for stem in keyword_stems:
                        mention_count += text.count(stem)
                    total_mentions += mention_count
            
            if posts_with_keyword > 0:
                keyword_stats.append({
                    'keyword': keyword,
                    'posts_count': posts_with_keyword,
                    'total_mentions': total_mentions,
                    'avg_per_post': total_mentions / posts_with_keyword if posts_with_keyword > 0 else 0
                })
        
        if not keyword_stats:
            print(f"  Нет данных по ключевым словам за {year} год")
            continue
        
        # Сортируем и строим график
        keyword_stats.sort(key=lambda x: x['posts_count'], reverse=True)
        top_keywords = keyword_stats[:15]
        
        # Создаем пузырьковую диаграмму
        fig, ax = plt.subplots(figsize=(14, 8))
        
        keywords = [k['keyword'][:20] + '...' if len(k['keyword']) > 20 else k['keyword'] 
                   for k in top_keywords]
        x_pos = range(len(keywords))
        y_values = [k['posts_count'] for k in top_keywords]
        
        mention_counts = [k['total_mentions'] for k in top_keywords]
        max_mentions = max(mention_counts) if mention_counts else 1
        bubble_sizes = [(m / max_mentions) * 1000 + 50 for m in mention_counts]
        
        colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(top_keywords)))[::-1]
        
        scatter = ax.scatter(x_pos, y_values, s=bubble_sizes, 
                           c=colors, alpha=0.6, edgecolors='black', linewidth=1)
        
        for i, (x, y, mentions, posts) in enumerate(zip(x_pos, y_values, mention_counts, [k['posts_count'] for k in top_keywords])):
            ax.annotate(f'{mentions} упом.\n({posts} постов)',
                       xy=(x, y),
                       xytext=(0, 10), textcoords="offset points",
                       ha='center', fontsize=8,
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
        
        ax.set_xlabel('Ключевые слова')
        ax.set_ylabel('Количество постов с ключевым словом')
        ax.set_title(f'Частота ключевых слов фактора "Связь с бизнесом" в ЗТТиЭ ({year})')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(keywords, rotation=45, ha='right', fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')
        
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Градация (от большего к меньшему)')
        
        plt.tight_layout()
        filename = f'пузырьковая_ЗТТиЭ_ключевые_слова_{year}.png'
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  Пузырьковая диаграмма для ЗТТиЭ сохранена: {filename}")
        
        stats_df = pd.DataFrame(keyword_stats)
        stats_df.to_csv(f'статистика_ключевых_слов_{year}.csv', index=False, encoding='utf-8-sig')

def main():
    print("="*80)
    print("АНАЛИЗ ФАКТОРОВ ПРИВЛЕКАТЕЛЬНОСТИ УЧЕБНЫХ ЗАВЕДЕНИЙ")
    print("="*80)
    
    results = []
    
    # Обрабатываем все файлы
    for source, years_dict in FILE_INFO.items():
        for year, filename in years_dict.items():
            print(f"\nОбработка: {source} ({year}) - {filename}")
            result = process_file(filename, source, year)
            if result:
                results.append(result)
    
    # Агрегируем результаты ЗТТиЭ
    aggregated_ztte = aggregate_ztte_results(results)
    
    # Собираем данные для Аносовского колледжа
    anosov_data = {}
    for r in results:
        if r and r['source'] == 'Аносовский колледж':
            anosov_data[r['year']] = r
    
    # Объединяем все данные
    all_data = {}
    for year, data in aggregated_ztte.items():
        all_data[f"ЗТТиЭ_{year}"] = data
    for year, data in anosov_data.items():
        all_data[f"Аносовский_{year}"] = data
    
    if not all_data:
        print("\nНЕТ ДАННЫХ ДЛЯ АНАЛИЗА!")
        return
    
    # Рассчитываем проценты (долю постов по каждому фактору)
    scored_data = calculate_factor_scores(all_data)
    
    # Создаем DataFrame для вывода
    rows = []
    for key, item in scored_data.items():
        row = {
            'Учреждение': 'ЗТТиЭ' if 'ЗТТиЭ' in key else 'Аносовский колледж',
            'Год': item['year'],
            'Всего постов': item['total_posts']
        }
        for factor, score in item['factor_scores'].items():
            # Красивые названия для вывода
            factor_names = {
                'экономические': 'Экономические, %',
                'социально_психологические': 'Социально-психол., %',
                'образовательные': 'Образовательные, %',
                'взаимодействие_с_бизнесом': 'Связь с бизнесом, %',
                'медийная_эффективность': 'Ср. ER Post'
            }
            row[factor_names.get(factor, factor)] = score
        rows.append(row)
    
    df_results = pd.DataFrame(rows)
    df_results = df_results.sort_values(['Учреждение', 'Год'])
    
    print("\n" + "="*80)
    print("РЕЗУЛЬТАТЫ АНАЛИЗА (доля постов по каждому фактору в %)")
    print("="*80)
    print(df_results.to_string(index=False))
    
    # Сохраняем в Excel
    df_results.to_excel('анализ_факторов_проценты.xlsx', index=False)
    print("\nРезультаты сохранены в файл 'анализ_факторов_проценты.xlsx'")
    
    # Строим сравнительные диаграммы для всех годов
    print("\n" + "="*80)
    print("СОЗДАНИЕ ГРАФИКОВ:")
    print("="*80)
    
    for year in ['2023', '2024', '2025']:
        plot_comparison_by_year(df_results, year)
    
    # Строим пузырьковые диаграммы для ЗТТиЭ (только фактор "Связь с бизнесом")
    # !!! ИСПРАВЛЕНО: передаем все три аргумента !!!
    plot_ztte_business_bubble(df_results, results, factor_stems)
    
    # Строим линейные графики динамики
    plot_results(df_results)

def plot_results(df):
    """Строит линейные графики для визуализации динамики"""
    
    # Подготовка данных для графиков
    ztte_data = df[df['Учреждение'] == 'ЗТТиЭ'].sort_values('Год')
    anosov_data = df[df['Учреждение'] == 'Аносовский колледж'].sort_values('Год')
    
    if ztte_data.empty and anosov_data.empty:
        print("Нет данных для построения линейных графиков")
        return
    
    # Факторы для графиков (проценты)
    factors_pct = ['Экономические, %', 'Социально-психол., %', 'Образовательные, %', 
                   'Связь с бизнесом, %']
    
    # 1. Линейные графики для процентных факторов
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Динамика факторов привлекательности (доля постов, %)', fontsize=14, fontweight='bold')
    
    colors_ztte = 'coral'
    colors_anosov = 'skyblue'
    
    for i, factor in enumerate(factors_pct[:4]):
        row = i // 2
        col = i % 2
        ax = axes[row, col]
        
        ztte_values = ztte_data[factor].values if not ztte_data.empty else []
        anosov_values = anosov_data[factor].values if not anosov_data.empty else []
        years = ztte_data['Год'].values if not ztte_data.empty else anosov_data['Год'].values
        
        if len(ztte_values) > 0:
            ax.plot(years, ztte_values, marker='o', linewidth=2, 
                   label='ЗТТиЭ', color=colors_ztte)
        if len(anosov_values) > 0:
            ax.plot(years, anosov_values, marker='s', linewidth=2, 
                   label='Аносовский колледж', color=colors_anosov)
        
        ax.set_xlabel('Год')
        ax.set_ylabel('Доля постов, %')
        ax.set_title(factor.replace(', %', ''))
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Определяем максимальное значение для оси Y
        max_val = 0
        if len(ztte_values) > 0:
            max_val = max(max_val, ztte_values.max())
        if len(anosov_values) > 0:
            max_val = max(max_val, anosov_values.max())
        ax.set_ylim(0, max_val * 1.1 if max_val > 0 else 10)
    
    plt.tight_layout()
    plt.savefig('динамика_факторов_проценты.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("  Линейные графики сохранены: динамика_факторов_проценты.png")
    
    # 2. Отдельный график для медийной эффективности
    fig, ax = plt.subplots(figsize=(10, 5))
    
    ztte_er = ztte_data['Ср. ER Post'].values if not ztte_data.empty else []
    anosov_er = anosov_data['Ср. ER Post'].values if not anosov_data.empty else []
    
    # Используем корректные переменные
    if not ztte_data.empty:
        years = ztte_data['Год'].values
    else:
        years = anosov_data['Год'].values
    
    if len(ztte_er) > 0:
        ax.plot(years, ztte_er, marker='o', linewidth=2, label='ЗТТиЭ', color='coral')
    if len(anosov_er) > 0:
        ax.plot(years, anosov_er, marker='s', linewidth=2, label='Аносовский колледж', color='skyblue')
    
    ax.set_xlabel('Год')
    ax.set_ylabel('Средний ER Post')
    ax.set_title('Медийная эффективность (ER Post)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('динамика_медийной_эффективности.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("  График медийной эффективности сохранен: динамика_медийной_эффективности.png")

if __name__ == "__main__":
    main()

СТЕМЫ КЛЮЧЕВЫХ СЛОВ:
Фактор 'экономические': 26 стемов
Фактор 'социально_психологические': 27 стемов
Фактор 'образовательные': 18 стемов
Фактор 'взаимодействие_с_бизнесом': 24 стемов
АНАЛИЗ ФАКТОРОВ ПРИВЛЕКАТЕЛЬНОСТИ УЧЕБНЫХ ЗАВЕДЕНИЙ

Обработка: ЗТТиЭ (2023) - ZTTiE-23.xlsx

  Файл ZTTiE-23.xlsx загружен. Постов: 568

  РЕЗУЛЬТАТЫ ЗТТиЭ (2023):
    Всего постов с текстом: 557
    ER средний: 0.906
    - экономические: 283 постов (50.8%)
    - социально_психологические: 367 постов (65.9%)
    - образовательные: 297 постов (53.3%)
    - взаимодействие_с_бизнесом: 343 постов (61.6%)

Обработка: ЗТТиЭ (2024) - ZTTiE-24.xlsx

  Файл ZTTiE-24.xlsx загружен. Постов: 729

  РЕЗУЛЬТАТЫ ЗТТиЭ (2024):
    Всего постов с текстом: 717
    ER средний: 1.845
    - экономические: 358 постов (49.9%)
    - социально_психологические: 489 постов (68.2%)
    - образовательные: 355 постов (49.5%)
    - взаимодействие_с_бизнесом: 363 постов (50.6%)

Обработка: ЗТТиЭ (2025) - ZTTiE-25.xlsx

  Файл ZTTiE-25.x